In [1]:
# 1. CODON TABLES SETUP
# Standard Genetic Code (genetic codon table - for Gene A and Gene C)
standard_codon_table = {
"ATT":"I", "ATC":"I", "ATA":"I", "ATG":"M","ACT":"T", "ACC":"T", "ACA":"T", "ACG":"T",
"AAT":"N", "AAC":"N", "AAA":"K", "AAG":"K","AGT":"S", "AGC":"S", "AGA":"R", "AGG":"R",

"GTT":"V", "GTC":"V", "GTA":"V", "GTG":"V","GCT":"A", "GCC":"A", "GCA":"A", "GCG":"A",
"GAT":"D", "GAC":"D", "GAA":"E", "GAG":"E","GGT":"G", "GGC":"G", "GGA":"G", "GGG":"G",

"TTT":"F" , "TTC":"F", "TTA":"L", "TTG":"L", "TCT":"S", "TCC":"S", "TCA":"S", "TCG":"S",
"TAT":"Y", "TAC":"Y", "TAA":"*", "TAG":"*", "TGT":"C", "TGC":"C", "TGA":"*", "TGG":"W",

"CTT":"L", "CTC":"L", "CTA":"L", "CTG":"L", "CCT":"P", "CCC":"P", "CCA":"P", "CCG":"P",
"CAT":"H", "CAC":"H", "CAA":"Q", "CAG":"Q","CGT":"R", "CGC":"R", "CGA":"R", "CGG":"R",
}

# Gene B  (Mitochondrial Table 2)
geneb_reverse_codes = {
    'F': ['TTT', 'TTC'],
    'L': ['TTA', 'TTG', 'CTT', 'CTC', 'CTA', 'CTG'],
    'I': ['ATT', 'ATC'],
    'M': ['ATA', 'ATG'],
    'V': ['GTT', 'GTC', 'GTA', 'GTG'],
    'S': ['TCT', 'TCC', 'TCA', 'TCG', 'AGT', 'AGC'],
    'P': ['CCT', 'CCC', 'CCA', 'CCG'],
    'T': ['ACT', 'ACC', 'ACA', 'ACG'],
    'A': ['GCT', 'GCC', 'GCA', 'GCG'],
    'Y': ['TAT', 'TAC'],
    'H': ['CAT', 'CAC'],
    'Q': ['CAA', 'CAG'],
    'N': ['AAT', 'AAC'],
    'K': ['AAA', 'AAG'],
    'D': ['GAT', 'GAC'],
    'E': ['GAA', 'GAG'],
    'C': ['TGT', 'TGC'],
    'W': ['TGA', 'TGG'],
    'R': ['CGT', 'CGC', 'CGA', 'CGG'],
    'G': ['GGT', 'GGC', 'GGA', 'GGG'],
    '*': ['TAA', 'TAG', 'AGA', 'AGG']
}

mito_codon_table = {} #Create mitochondrial codon table
for aa, codons in geneb_reverse_codes.items():
    for codon in codons:
        mito_codon_table[codon] = aa

# 2. DATA SETUP
# Store gene-specific information for the pipeline

genes_info = {
    "genea": {"name": "SEPHS2",
        "category": "Selenoprotein (A)",
        "fasta_file": "../data/genea.fasta",
        "pro_file": "../data/genea_pro.fasta",
        "cds_coords": (147, 1494),  # 148:1494 (1-based to 0-based start-1:end)
        "table": standard_codon_table},
    "geneb": {"name": "MT-ND3",
        "category": "Mitochondrial (B)",
        "fasta_file": "../data/geneb.fasta",
        "pro_file": "../data/geneb_pro.fasta",
        "cds_coords": (10058, 10404),  # 10059:10404 (start-1: end)
        "table": mito_codon_table},
    "genec": {"name": "PPIB",
        "category": "Nuclear (C)",
        "fasta_file": "../data/genec.fasta",
        "pro_file": "../data/genec_pro.fasta",
        "cds_coords": (36, 687),  # 37:687 (start-1: end)
        "table": standard_codon_table}}

# Read nucleotide and protein sequence from FASTA
def read_fasta_seq(filepath):
    """Loads a FASTA file using plain Python string operations."""
    sequence = ""
    with open(filepath, "r") as data:
        for line in data:
            if line[0] != ">":  # Skip the fasta header line
                sequence += line.replace("\n", "")  # Remove newlines and append
    return sequence
    
# Translate CDS using the codon table  
def translate_cds(cds_seq, table):
    """Translates CDS sequence using specified codon lookup structure."""
    protein = ""
# Read the CDS three bases at a time
    for i in range(0, len(cds_seq), 3):
        codon = cds_seq[i:i+3]
        if len(codon) == 3:
            protein += table.get(codon, "?")
    return protein

# 3. PIPELINE (SUMMARY TABLE)
# Store results for all three genes
summary_results = []

# Process each gene using its stored information
for gene_key, info in genes_info.items():

# Read mRNA and deposited protein sequences
    mrna_seq = read_fasta_seq(info["fasta_file"])
    dep_protein_seq = read_fasta_seq(info["pro_file"])

# Extract the CDS using the recorded range
    start, end = info["cds_coords"]
    cds_seq = mrna_seq[start:end]
    cds_length = len(cds_seq)

# Translate CDS using the genetic code
    trans_protein = translate_cds(cds_seq, info["table"])
    
# Remove terminal stop codon for protein length comparison
    if trans_protein.endswith("*"):
        trans_protein_clean = trans_protein[:-1]
    else:
        trans_protein_clean = trans_protein    
 # Calculate translated and deposited protein lengths
    trans_len = len(trans_protein_clean)
    dep_len = len(dep_protein_seq)

# Compare sequences by positions
    identical = "Y"
    mismatch_pos = "N/A"

    if trans_protein_clean == dep_protein_seq:
        identical = "Y"
    else:
        identical = "N"
# Find the length of the shorter protein sequence
        if len(trans_protein_clean) < len(dep_protein_seq): 
            min_len = len(trans_protein_clean)
        else: 
            min_len = len(dep_protein_seq)
# Find the first position where the sequences differ
        for i in range(min_len):
            if trans_protein_clean[i] != dep_protein_seq[i]:
                mismatch_pos = str(i + 1)
                break
# Record the position if the difference is due to sequence length
        if mismatch_pos == "N/A" and trans_len != dep_len:
            mismatch_pos = str(min_len + 1)
            
 # Store the results for the current gene
    summary_results.append({
        "Gene Name": info["name"],
        "Category": info["category"],
        "CDS Length": cds_length,
        "Translated Length": trans_len,
        "Deposited Length": dep_len,
        "Identical (Y/N)": identical,
        "First Mismatch Position": mismatch_pos
    })

#4 SUMMARY TABLE

with open("../results/summary_table.csv", "w") as output:

# Write column headings

    output.write(
        "Gene Name,Category,CDS Length,Translated Length,"
        "Deposited Length,Identical (Y/N),First Mismatch Position\n"
    )

# Write the results for each gene
    for result in summary_results:
        output.write(
            result["Gene Name"] + "," +
            result["Category"] + "," +
            str(result["CDS Length"]) + "," +
            str(result["Translated Length"]) + "," +
            str(result["Deposited Length"]) + "," +
            result["Identical (Y/N)"] + "," +
            str(result["First Mismatch Position"]) + "\n"
        )
# Final print of output message and it's path
print("Pipeline executed successfully.")
print("Output saved to: ../results/summary_table.csv")

Pipeline executed successfully.
Output saved to: ../results/summary_table.csv
